In [130]:
import os
import PyPDF2
import numpy as np
import pandas as pd
from tabula.io import read_pdf
from datetime import datetime
import re

In [131]:
file_name = r"C:\Users\admin\Downloads\05.02.2021 £2274.05 Chargecooler Covers [0001].pdf"

r"C:\Users\admin\Downloads\05.02.2021 £2274.05 Chargecooler Covers [0001].pdf"

'C:\\Users\\admin\\Downloads\\05.02.2021 £2274.05 Chargecooler Covers [0001].pdf'

In [132]:
invoice_type = "Products"

input_file = fr"C:\Users\admin\Downloads\09.05.2023 £600.12 ChargeCoolerCovers.pdf"

In [133]:
name = "ChargeCoolerCovers"

table1 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(154,380,254,532),
                  columns=[532],
                  pandas_options={'header': None},
                  encoding="windows-1254")

heading = table1[0]
display(heading)

docnum = heading[0][0].split(':')[1].strip()
print(docnum)

date = heading[0][1].split(':')[1].strip()
date = str(datetime.strptime(date, "%d/%m/%Y"))
print(date)

ordernum = None

transfernum = None
print(ordernum)
print(transfernum)


,0
0,Invoice #: 0015
1,Invoice Date: 09/05/2023


0015
2023-05-09 00:00:00
None
None


In [134]:
table2 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(280,60,830,537),
                  columns=[113,231,306,380,457,537],
                  pandas_options={'header': None},
                  encoding="windows-1254")
content=table2[0]

content

,0,1,2,3,4,5
0,MLP113,M3 Competition - vertical,5,£33.34,azzzzzzzzzzzzzz,£166.70
1,MLP117,M4 CS - vertical,5,£33.34,NaN,$166.70
2,MLP118,M4 Heritage - Horizontal,5,£33.34,NaN,$166.70
3,NaN,NaN,NaN,NaN,Invoice Subtotal,£500.10
4,NaN,NaN,NaN,NaN,Tax Rate,20.00%
5,NaN,NaN,NaN,NaN,V.A.T.,£100.02
6,NaN,NaN,NaN,NaN,Other,NaN
7,NaN,NaN,NaN,NaN,Deposit Received,NaN
8,Make payable,to Nigel Purvis,NaN,NaN,NaN,NaN
9,Account Numbe,r 08996084 Sort Code 938621 IBAN GB6,7FTBK93862108996084,BIC FTBKGB2B,NaN,NaN


In [135]:
content[2] = pd.to_numeric(content[2], errors='coerce')  # Remove anything that not number in column 2
content = content.dropna(subset=[2,5]).reset_index(drop=True) # Remove rows with NaN in column 2

content[[3,5]] = content[[3,5]].replace('[£,$, ]','', regex=True).astype('float64')
content

,0,1,2,3,4,5
0,MLP113,M3 Competition - vertical,5.0,33.34,azzzzzzzzzzzzzz,166.7
1,MLP117,M4 CS - vertical,5.0,33.34,NaN,166.7
2,MLP118,M4 Heritage - Horizontal,5.0,33.34,NaN,166.7


In [136]:
content.rename(columns={
    0: 'Item #',
    1: 'Description',
    2: 'Qty',
    3: 'Unit Price',
    4: 'Discount',
    5: 'Price'}, inplace=True)

display(content)

,Item #,Description,Qty,Unit Price,Discount,Price
0,MLP113,M3 Competition - vertical,5.0,33.34,azzzzzzzzzzzzzz,166.7
1,MLP117,M4 CS - vertical,5.0,33.34,NaN,166.7
2,MLP118,M4 Heritage - Horizontal,5.0,33.34,NaN,166.7


In [137]:
dict_content = content.to_dict(orient='records')
dict_content

line_items=[]
for item in dict_content:

    partNum = item['Item #']
    desc = item['Description']
    quantity = item['Qty']
    netTotal = item['Price']

    if partNum == 'No code':
        partNum = None

    print(partNum)
    
    line_item = {"line_type": "inventory",
                        "sku": partNum,
                        "name": desc,
                        "quantity": int(quantity),
                        "net_total": float(netTotal),
                        "tax_type": "INPUT2"}
    
    line_items.append(line_item)
    
print(line_items)

MLP113
MLP117
MLP118
[{'line_type': 'inventory', 'sku': 'MLP113', 'name': 'M3 Competition - vertical', 'quantity': 5, 'net_total': 166.7, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MLP117', 'name': 'M4 CS - vertical', 'quantity': 5, 'net_total': 166.7, 'tax_type': 'INPUT2'}, {'line_type': 'inventory', 'sku': 'MLP118', 'name': 'M4 Heritage - Horizontal', 'quantity': 5, 'net_total': 166.7, 'tax_type': 'INPUT2'}]


In [138]:
table3 = read_pdf(input_file,
                  pages="1",
                  silent=True,
                  guess=False,
                  area=(304,381,830,537),
                  columns=[457,533],
                  pandas_options={'header': None},
                  encoding='windows-1254')

total_content=table3[0]
display(total_content)

#To remove "£"
total_content[[1]] = total_content[[1]].replace('[£,$, ]','', regex=True).astype('string')
total_content = total_content.dropna(subset=[0]).reset_index(drop=True) # Remove rows with NaN in column 0
display(total_content)


,0,1
0,NaN,$166.70
1,NaN,$166.70
2,Invoice Subtotal,£500.10
3,Tax Rate,20.00%
4,V.A.T.,£100.02
5,Other,NaN
6,Deposit Received,NaN
7,TOTAL,£600.12


,0,1
0,Invoice Subtotal,500.10
1,Tax Rate,20.00%
2,V.A.T.,100.02
3,Other,<NA>
4,Deposit Received,<NA>
5,TOTAL,600.12


In [139]:
row_index = total_content.index[total_content[0] == "TOTAL"].tolist()[0]
final_total = str(total_content[1][row_index])
final_total

'600.12'

In [140]:
payload = {}
keys = ["Source File",
        "Type",
        "Name",
        "Date",
        "Reference No.",
        "Order No.",
        "Transfer No.",
        "Document No.",
        "Line Items",
        "Total"]

values = [file_name,
        invoice_type,
        name,
        date,
        docnum,
        ordernum,
        transfernum,
        None,
        line_items,
        final_total]

for i, key in enumerate(keys):
    payload[key] = values[i]

payload

{'Source File': 'C:\\Users\\admin\\Downloads\\05.02.2021 £2274.05 Chargecooler Covers [0001].pdf',
 'Type': 'Products',
 'Name': 'ChargeCoolerCovers',
 'Date': '2023-05-09 00:00:00',
 'Reference No.': '0015',
 'Order No.': None,
 'Transfer No.': None,
 'Document No.': None,
 'Line Items': [{'line_type': 'inventory',
   'sku': 'MLP113',
   'name': 'M3 Competition - vertical',
   'quantity': 5,
   'net_total': 166.7,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MLP117',
   'name': 'M4 CS - vertical',
   'quantity': 5,
   'net_total': 166.7,
   'tax_type': 'INPUT2'},
  {'line_type': 'inventory',
   'sku': 'MLP118',
   'name': 'M4 Heritage - Horizontal',
   'quantity': 5,
   'net_total': 166.7,
   'tax_type': 'INPUT2'}],
 'Total': '600.12'}